In [ ]:
### GOAL: generate a .yaml or readable pmapping from the saved .pkl file
# use this to calculate CI of various components of spec decoding workload

### ALL UNFUSED

In [1]:
from pathlib import Path
import contextlib
import io
import json
import pickle

import pandas as pd

try:
    import yaml
except ImportError:
    yaml = None

try:
    from IPython.display import display
except ImportError:
    display = print

# pull in the pmapping generated by milestone2_runs.ipynb.

SAVED_MAPPING = Path("saved_mappings/spec_target_ctx128_gamma3.pkl")

if not SAVED_MAPPING.exists():
    raise FileNotFoundError("Could not find spec_target_ctx128_gamma3.pkl")

OUTPUT_DIR = SAVED_MAPPING.parent
READABLE_YAML = OUTPUT_DIR / f"{SAVED_MAPPING.stem}.readable.yaml"
RENDER_TXT = OUTPUT_DIR / f"{SAVED_MAPPING.stem}.render.txt"
COMPUTE_PATHS_CSV = OUTPUT_DIR / f"{SAVED_MAPPING.stem}.compute_paths.csv"
MATMUL_CI_CSV = OUTPUT_DIR / f"{SAVED_MAPPING.stem}.matmul_intensity.csv"
COMPONENT_CI_CSV = OUTPUT_DIR / f"{SAVED_MAPPING.stem}.component_intensity.csv"

print(f"Loading mapping from: {SAVED_MAPPING.resolve()}")
print(f"Pickle size: {SAVED_MAPPING.stat().st_size:,} bytes")

with open(SAVED_MAPPING, "rb") as f:
    mapping_obj = pickle.load(f)

print(f"Loaded object type: {type(mapping_obj).__module__}.{type(mapping_obj).__name__}")


Loading mapping from: /home/workspace/workspace/saved_mappings/spec_target_ctx128_gamma3.pkl
Pickle size: 142,895 bytes
Loaded object type: accelforge.frontend.mapping.mapping.Mapping


In [2]:
def simple_value(value):
    """We convert AccelForge objects to readable scalar/list values for YAML export."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (list, tuple)):
        return [simple_value(v) for v in value]
    if isinstance(value, (set, frozenset)):
        return sorted(simple_value(v) for v in value)
    return str(value)

NODE_ATTRS = (
    "tensors",
    "component",
    "einsum",
    "rank_variable",
    "tile_shape",
    "initial_tile_shape",
    "name",
    "persistent",
    "resource",
    "purposes",
)


def summarize_node(node):
    """Keep the structural mapping fields and drop large hardware-model internals."""
    summary = {"type": type(node).__name__}

    for attr in NODE_ATTRS:
        if hasattr(node, attr):
            try:
                summary[attr] = simple_value(getattr(node, attr))
            except Exception as exc:
                summary[attr] = f"<unreadable: {exc}>"

    child_nodes = getattr(node, "nodes", None)
    if child_nodes:
        summary["nodes"] = [summarize_node(child) for child in child_nodes]
    return summary


mapping_nodes = getattr(mapping_obj, "nodes", [])
mapping_summary = {
    "note": "Readable summary for analysis only (not guaranteed to be an AccelForge-loadable YAML mapping).",
    "source_pickle": str(SAVED_MAPPING),
    "root_type": type(mapping_obj).__name__,
    "nodes": [summarize_node(node) for node in mapping_nodes],
}

if yaml is not None:
    READABLE_YAML.write_text(yaml.safe_dump(mapping_summary, sort_keys=False, width=120))
else:
    READABLE_YAML.write_text(json.dumps(mapping_summary, indent=2))

# Capture render() if AccelForge prints it; otherwise fall back to repr(mapping_obj).
render_buffer = io.StringIO()
try:
    with contextlib.redirect_stdout(render_buffer):
        rendered = mapping_obj.render()
    rendered_text = render_buffer.getvalue()
    if rendered is not None:
        rendered_text += str(rendered)
    if not rendered_text.strip():
        rendered_text = repr(mapping_obj)
except Exception as exc:
    rendered_text = f"mapping_obj.render() failed: {exc}\n\nrepr fallback:\n{repr(mapping_obj)}"

RENDER_TXT.write_text(rendered_text)

print(f"Saved readable mapping summary to: {READABLE_YAML}")
print(f"Saved render/repr text to: {RENDER_TXT}")


def node_label(node):
    label_parts = [type(node).__name__]
    for attr in ("einsum", "component", "rank_variable", "tile_shape", "name"):
        if hasattr(node, attr):
            value = getattr(node, attr)
            if value is not None:
                label_parts.append(f"{attr}={value}")
    return "(" + ", ".join(label_parts) + ")"


def iter_compute_nodes(node, path=()):
    current_path = path + (node_label(node),)

    if type(node).__name__ == "Compute":
        yield {
            "einsum": getattr(node, "einsum", None),
            "component": getattr(node, "component", None),
            "mapping_path": " -> ".join(current_path),
        }

    for child in getattr(node, "nodes", []) or []:
        yield from iter_compute_nodes(child, current_path)


compute_rows = []
for root_idx, root_node in enumerate(mapping_nodes):
    compute_rows.extend(iter_compute_nodes(root_node, (f"root[{root_idx}]",)))

compute_paths_df = pd.DataFrame(compute_rows)
compute_paths_df.to_csv(COMPUTE_PATHS_CSV, index=False)
print(f"Saved compute-node paths to: {COMPUTE_PATHS_CSV}")
display(compute_paths_df)


Saved readable mapping summary to: saved_mappings/spec_target_ctx128_gamma3.readable.yaml
Saved render/repr text to: saved_mappings/spec_target_ctx128_gamma3.render.txt
Saved compute-node paths to: saved_mappings/spec_target_ctx128_gamma3.compute_paths.csv


,einsum,component,mapping_path
0,I,ScalarUnit,root[2] -> (Sequential) -> (Nested) -> (Comput...
1,V_new,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
2,K_new,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
3,Q_new,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
4,QK,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
5,QK_softmax,ScalarUnit,root[2] -> (Sequential) -> (Nested) -> (Comput...
6,AV,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
7,Z,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
8,FFA,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
9,FFB,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...


In [11]:
# For this specific case, saved pickle is the pmapping generated with context_len=128 and gamma=3.
# In true_validator_spec(), BATCH_SIZE=1, N_NEW_TOKENS=gamma, and N_TOKENS=context_len+gamma.

CONTEXT_LEN = 128
GAMMA = 3
BITS_PER_VALUE = 8

'''
B: batch size
M: number of new/query tokens (lookahead)
M_FULL: full context length (context_len + gamma)
H: number of attention heads (from GPT)
E: per-head key/query dimension (from GPT)
F: per-head value dimension (from GPT)
D: model hidden dimension
C: feed-forward hidden dimension
J: output hidden dimension
G: projection hidden dimension
'''

R = {
    "B": 1,
    "M": GAMMA,
    "M_FULL": CONTEXT_LEN + GAMMA,
    "H": 96,
    "E": 128,
    "F": 128,
}

# these are all derived from the GPT architecture
R["D"] = R["H"] * R["E"]
R["C"] = 4 * R["H"] * R["E"]
R["J"] = R["H"] * R["E"]
R["G"] = R["H"] * R["E"]


def matmul_comp_intensity_row(einsum, stage, logical_matmul, ops_macs, lhs_read_values, rhs_read_values, output_write_values):
    total_read_write_values = lhs_read_values + rhs_read_values + output_write_values # something like M*N+N*K+M*K for a standard m,k x k,n matmul
    bytes_at_8bit = total_read_write_values * BITS_PER_VALUE / 8
    return {
        "einsum": einsum,
        "stage": stage,
        "logical_matmul": logical_matmul,
        "ops_macs": ops_macs,
        "lhs_read_values": lhs_read_values,
        "rhs_read_values": rhs_read_values,
        "output_write_values": output_write_values,
        "read_write_values": total_read_write_values,
        "bytes_at_8bit": bytes_at_8bit,
        "comp_intensity_macs_per_value": ops_macs / total_read_write_values,
        "comp_intensity_macs_per_byte": ops_macs / bytes_at_8bit,
    }


B, M, M_FULL, H, E, F, D, C, J, G = (R[k] for k in ("B", "M", "M_FULL", "H", "E", "F", "D", "C", "J", "G"))

# these are all taken from the workload
matmul_rows = [
    # V_new[b, m, h, e] = I[b, m, d] * WV[h, e, d]
    matmul_comp_intensity_row(
        "V_new",
        "KV-cache value projection",
        "(B*M x D) @ (D x H*E) -> (B*M x H*E)",
        ops_macs = B * M * D * H * E,
        lhs_read_values = B * M * D,
        rhs_read_values = D * H * E,
        output_write_values = B * M * H * E,
    ),
    # K_new[b, m, h, e] = I[b, m, d] * WK[h, e, d]
    matmul_comp_intensity_row(
        "K_new",
        "KV-cache key projection",
        "(B*M x D) @ (D x H*E) -> (B*M x H*E)",
        ops_macs = B * M * D * H * E,
        lhs_read_values = B * M * D,
        rhs_read_values = D * H * E,
        output_write_values = B * M * H * E,
    ),
    # Q_new[b, m, h, e] = I[b, m, d] * WQ[h, e, d]
    matmul_comp_intensity_row(
        "Q_new",
        "Query projection",
        "(B*M x D) @ (D x H*E) -> (B*M x H*E)",
        ops_macs = B * M * D * H * E,
        lhs_read_values = B * M * D,
        rhs_read_values = D * H * E,
        output_write_values = B * M * H * E,
    ),
    # QK[b, m, m_full, h] = Q_new[b, m, h, e] * K[b, m_full, h, e]
    matmul_comp_intensity_row(
        "QK",
        "Attention score matmul",
        "B*H batches of (M x E) @ (E x M_FULL) -> (M x M_FULL)",
        ops_macs = B * H * M * M_FULL * E,
        lhs_read_values = B * H * M * E,
        rhs_read_values = B * H * M_FULL * E,
        output_write_values = B * H * M * M_FULL,
    ),
    # AV[b, m, h, f] = QK_softmax[b, m, m_full, h] * V[b, m_full, h, E: f]
    matmul_comp_intensity_row(
        "AV",
        "Attention-value matmul",
        "B*H batches of (M x M_FULL) @ (M_FULL x F) -> (M x F)",
        ops_macs = B * H * M * M_FULL * F,
        lhs_read_values = B * H * M * M_FULL,
        rhs_read_values = B * H * M_FULL * F,
        output_write_values = B * H * M * F,
    ),
    # Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]
    matmul_comp_intensity_row(
        "Z",
        "Attention output projection",
        "(B*M x H*F) @ (H*F x G) -> (B*M x G)",
        ops_macs = B * M * H * F * G,
        lhs_read_values = B * M * H * F,
        rhs_read_values = H * F * G,
        output_write_values = B * M * G,
    ),
    # FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]
    matmul_comp_intensity_row(
        "FFA",
        "Feed-forward expansion",
        "(B*M x G) @ (G x C) -> (B*M x C)",
        ops_macs = B * M * G * C,
        lhs_read_values = B * M * G,
        rhs_read_values = G * C,
        output_write_values = B * M * C,
    ),
    # FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]
    matmul_comp_intensity_row(
        "FFB",
        "Feed-forward projection",
        "(B*M x C) @ (C x J) -> (B*M x J)",
        ops_macs = B * M * C * J,
        lhs_read_values = B * M * C,
        rhs_read_values = C * J,
        output_write_values = B * M * J,
    ),
]

non_matmul_rows = [
    # I[b, m, d] = I_in[b, m, d]
    # The CI here is just 0 because it's a copy step (purely memory-dependent)
    {
        "einsum": "I",
        "stage": "Input copy / staging",
        "operation_model": "copy",
        "ops": 0,
        "read_values": B * M * D,
        "write_values": B * M * D,
    },
    # QK_softmax[b, m, m_full, h] = QK[b, m, m_full, h]
    {
        "einsum": "QK_softmax",
        "stage": "Attention softmax",
        "operation_model": "max + exp + sum + divide over M_FULL",
        "ops": B * M * H * M_FULL * 4,
        "read_values": B * M * H * M_FULL,
        "write_values": B * M * H * M_FULL,
    },
]

matmul_comp_intensity_df = pd.DataFrame(matmul_rows)

non_matmul_df = pd.DataFrame(non_matmul_rows)
non_matmul_df["read_write_values"] = (
    non_matmul_df["read_values"] + non_matmul_df["write_values"]
)
non_matmul_df["comp_intensity_ops_per_value"] = (
    non_matmul_df["ops"] / non_matmul_df["read_write_values"]
)

if not compute_paths_df.empty:
    compute_lookup = compute_paths_df.drop_duplicates("einsum")[["einsum", "component", "mapping_path"]]
    matmul_comp_intensity_df = matmul_comp_intensity_df.merge(compute_lookup, on="einsum", how="left")

matmul_comp_intensity_df = matmul_comp_intensity_df.sort_values("comp_intensity_macs_per_value").reset_index(drop=True)
matmul_comp_intensity_df.to_csv(MATMUL_CI_CSV, index=False)
print(f"Saved matmul computational intensity table to: {MATMUL_CI_CSV}")
display(matmul_comp_intensity_df)

if "component" in matmul_comp_intensity_df.columns:
    component_comp_intensity_df = (
        matmul_comp_intensity_df.groupby("component", dropna=False)
        .agg(
            total_ops_macs=("ops_macs", "sum"),
            total_read_write_values=("read_write_values", "sum"),
            total_bytes_at_8bit=("bytes_at_8bit", "sum"),
            matmul_count=("einsum", "count"),
        )
        .reset_index()
    )
    component_comp_intensity_df["comp_intensity_macs_per_value"] = component_comp_intensity_df["total_ops_macs"] / component_comp_intensity_df["total_read_write_values"]
    component_comp_intensity_df["comp_intensity_macs_per_byte"] = component_comp_intensity_df["total_ops_macs"] / component_comp_intensity_df["total_bytes_at_8bit"]
    component_comp_intensity_df.to_csv(COMPONENT_CI_CSV, index=False)
    print(f"Saved component-level computational intensity table to: {COMPONENT_CI_CSV}")
    display(component_comp_intensity_df)


matmul_einsums = set(matmul_comp_intensity_df["einsum"])
all_compute_einsums = set(compute_paths_df["einsum"]) if not compute_paths_df.empty else set()
non_matmul_einsums = sorted(all_compute_einsums - matmul_einsums)
print("Non-matmul compute nodes excluded from matmul CI:", non_matmul_einsums)
print("Rank sizes used:", R)

display(non_matmul_df)

Saved matmul computational intensity table to: saved_mappings/spec_target_ctx128_gamma3.matmul_intensity.csv


,einsum,stage,logical_matmul,ops_macs,lhs_read_values,rhs_read_values,output_write_values,read_write_values,bytes_at_8bit,comp_intensity_macs_per_value,comp_intensity_macs_per_byte,component,mapping_path
0,QK,Attention score matmul,B*H batches of (M x E) @ (E x M_FULL) -> (M x ...,4829184,36864,1609728,37728,1684320,1684320.0,2.867142,2.867142,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
1,AV,Attention-value matmul,B*H batches of (M x M_FULL) @ (M_FULL x F) -> ...,4829184,37728,1609728,36864,1684320,1684320.0,2.867142,2.867142,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
2,V_new,KV-cache value projection,(B*M x D) @ (D x H*E) -> (B*M x H*E),452984832,36864,150994944,36864,151068672,151068672.0,2.998536,2.998536,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
3,K_new,KV-cache key projection,(B*M x D) @ (D x H*E) -> (B*M x H*E),452984832,36864,150994944,36864,151068672,151068672.0,2.998536,2.998536,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
4,Q_new,Query projection,(B*M x D) @ (D x H*E) -> (B*M x H*E),452984832,36864,150994944,36864,151068672,151068672.0,2.998536,2.998536,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
5,Z,Attention output projection,(B*M x H*F) @ (H*F x G) -> (B*M x G),452984832,36864,150994944,36864,151068672,151068672.0,2.998536,2.998536,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
6,FFA,Feed-forward expansion,(B*M x G) @ (G x C) -> (B*M x C),1811939328,36864,603979776,147456,604164096,604164096.0,2.999085,2.999085,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...
7,FFB,Feed-forward projection,(B*M x C) @ (C x J) -> (B*M x J),1811939328,147456,603979776,36864,604164096,604164096.0,2.999085,2.999085,MAC,root[2] -> (Sequential) -> (Nested) -> (Comput...


Saved component-level computational intensity table to: saved_mappings/spec_target_ctx128_gamma3.component_intensity.csv


,component,total_ops_macs,total_read_write_values,total_bytes_at_8bit,matmul_count,comp_intensity_macs_per_value,comp_intensity_macs_per_byte
0,MAC,5445476352,1815971520,1.815972e+09,8,2.998657,2.998657


Non-matmul compute nodes excluded from matmul CI: ['I', 'QK_softmax']
Rank sizes used: {'B': 1, 'M': 3, 'M_FULL': 131, 'H': 96, 'E': 128, 'F': 128, 'D': 12288, 'C': 49152, 'J': 12288, 'G': 12288}


,einsum,stage,operation_model,ops,read_values,write_values,read_write_values,comp_intensity_ops_per_value
0,I,Input copy / staging,copy,0,36864,36864,73728,0.0
1,QK_softmax,Attention softmax,max + exp + sum + divide over M_FULL,150912,37728,37728,75456,2.0
